# Lecture 7: Agent Consequences

An evidence interpreter is incomplete if nothing changes after evaluation. PACTA has a small consequence engine: it can build a lower-layer Rust decision capsule when evidence satisfies policy, and it refuses wallet construction when coverage is insufficient.


## Learning Objectives

- Explain how risk levels map to actions.
- Run a dry-run agent action.
- Understand the generated Rust capsule.
- Explain why R3 permits lower-layer use but denies wallet demos.
- Connect transparency receipts to build authorization.


## Action Policy

Current actions:

- `build-library`: default threshold R3. Produces a proof-gated component capsule.
- `build-wallet-demo`: threshold R4. Writes a denial artifact below R4.

This is deliberately conservative. The theorem boundary for Ed25519 field plus Edwards arithmetic is valuable, but it does not cover key custody, encoding, hashing, scalar arithmetic completeness, signature verification, transaction construction, or market decisions.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "src" / "pacta").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from pacta.agent import run_agent_action
from pacta.claims import build_claim_card
from pacta.config import load_config

repo = load_config(repo_root / "examples" / "repos.yaml").repo_named("dalek-ed25519-verified")
card = build_claim_card(repo, repo_root / "repos" / repo.name, offline_fixture=True)

library_decision = run_agent_action(card, "build-library", repo_root / "artifacts-notebook", dry_run=True)
wallet_decision = run_agent_action(card, "build-wallet-demo", repo_root / "artifacts-notebook", dry_run=True)
print(library_decision.to_dict())
print(wallet_decision.to_dict())


## The Rust Capsule

The generated capsule is not cryptographic code. It is a consumable policy artifact. It embeds the claim card and exposes constants such as:

- component,
- repo URL,
- kind,
- verified backend,
- risk level,
- evidence mode,
- attestation provider,
- deployment constraints.

Downstream automation can import this crate and check `allowed_for_lower_layer_crypto()` before enabling a code path.


In [ ]:
from pacta.artifact import _lib_rs

print(_lib_rs(card).splitlines()[:24])


## Receipt-Required Builds

The strongest dogfood path in this prototype is:

1. Provider replays Lean and signs attestation.
2. Provider appends attestation to a Merkle transparency log.
3. Provider emits an inclusion receipt with Signed Tree Head.
4. Agent verifies provider signature, receipt inclusion proof, and STH signature.
5. Agent builds only if risk and transparency policy pass.

This transforms "I read a report" into "I accepted a logged, signed proof-check result and acted within its theorem boundary."


## Command-Line Lab

Run these from the repository root after generating a provider attestation:

```bash
pacta receipt-verify \
  --attestation provider/out/dalek-ed25519.attestation.yaml \
  --receipt provider/out/dalek-ed25519.receipt.yaml \
  --log-public-key provider/state/local-provider/provider.ed25519.pub

pacta agent \
  --config examples/repos.yaml \
  --repo-name dalek-ed25519-verified \
  --repo repos/dalek-ed25519-verified \
  --attestation provider/out/dalek-ed25519.attestation.yaml \
  --trust-attestation-provider local-pacta-provider \
  --attestation-public-key provider/state/local-provider/provider.ed25519.pub \
  --transparency-receipt provider/out/dalek-ed25519.receipt.yaml \
  --transparency-log-public-key provider/state/local-provider/provider.ed25519.pub \
  --require-transparency-receipt \
  --action build-library
```


## Exercises

- Modify a claim card to R2 and show that `build-library` is refused.
- Explain why a denial artifact is useful for auditability.
- Design a policy where an agent requires `both` Ed25519 and ML-DSA signatures for production deployment but allows Ed25519-only in a local lab.
- Write a downstream Rust pseudo-code snippet that imports the generated capsule before enabling a code path.
